# CatBoost — Optuna Hyperparameter Tuning (GPU)

Same data pipeline / CV / sample_weight as `catboost.ipynb`. Searches CatBoost-specific knobs: `depth`, `learning_rate`, `l2_leaf_reg`, `bagging_temperature`, `border_count`, `random_strength`, `min_data_in_leaf`.

**Goal:** push CatBoost single-OOF from 0.93130 (untuned) toward the GBMs' ~0.935 so the 3-way blend pays off on the LB (sub_28 LB underperformed because untuned CB was too weak even though OOF said the blend should win).

After this finishes, paste `study.best_params` into `catboost.ipynb` and re-run it — that produces the OOF/test CSVs `blender3.ipynb` consumes.

## Imports

In [ ]:
import pandas as pd
import numpy as np
import warnings, json
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)

import catboost as cb
import optuna
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold

from common import *

print('catboost:', cb.__version__)
print('optuna:  ', optuna.__version__)

## Load + preprocess (mirror of catboost.ipynb)

In [ ]:
train_df = pd.read_csv('archive/train.csv')
test_df  = pd.read_csv('archive/test.csv')
orig_df  = pd.read_csv('archive/f1_strategy_dataset_v4.csv')

df = (train_df.pipe(copy_data).pipe(clean_data).pipe(remove_duplicates).pipe(make_new_features))
orig_df_cleaned = (orig_df.pipe(copy_data).pipe(clean_data).pipe(remove_duplicates).pipe(make_new_features))

train_df_length = df.shape[0]
df = pd.concat([df, orig_df_cleaned])
if 'normalized_tyrelife' in df.columns:
    df = df.drop('normalized_tyrelife', axis=1)
df = df.reset_index(drop=True)

sample_weights = np.ones(df.shape[0])
sample_weights[train_df_length:] = 1.25

target = get_target()
features = get_features(df)
categorical_features = ['compound', 'race', 'year_cat']
for c in categorical_features:
    df[c] = df[c].astype(str).fillna('NA')

X, y = df[features], df[target]
groups = (df['race'].astype(str) + '_' + df['year'].astype(str)).values
strat_y = (df['pitnextlap'].astype(str) + '_' + df['year'].astype(str))

print(df.shape, '/', len(features), 'features')

## Optuna objective — 5-fold StratifiedGroupKFold + early stopping

In [ ]:
TUNE_N_SPLITS = 5
SEED = 42

def objective(trial):
    params = {
        'iterations': 5000,
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'task_type': 'GPU',
        'devices': '0',
        'random_seed': 123,
        'verbose': 0,
        'learning_rate':       trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'depth':               trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg':         trial.suggest_float('l2_leaf_reg', 0.5, 30.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 2.0),
        'border_count':        trial.suggest_categorical('border_count', [64, 128, 254]),
        'random_strength':     trial.suggest_float('random_strength', 0.0, 5.0),
        'min_data_in_leaf':    trial.suggest_int('min_data_in_leaf', 1, 100),
    }

    sgkf = StratifiedGroupKFold(n_splits=TUNE_N_SPLITS, shuffle=True, random_state=123)
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(sgkf.split(X, strat_y, groups=groups)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = CatBoostClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=(X_val, y_val),
            cat_features=categorical_features,
            sample_weight=sample_weights[tr_idx],
            early_stopping_rounds=50,
            verbose=0,
        )
        proba = model.predict_proba(X_val)[:, 1]
        fold_scores.append(roc_auc_score(y_val, proba))

        trial.report(float(np.mean(fold_scores)), step=fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_scores))

## Run study

Enqueue the default-ish params as trial 0 to anchor the search.

In [ ]:
N_TRIALS = 60
TIMEOUT_SEC = 60 * 60 * 5   # 5 hours cap

baseline_params = {
    'learning_rate':       0.05,
    'depth':               6,
    'l2_leaf_reg':         3.0,
    'bagging_temperature': 1.0,
    'border_count':        254,
    'random_strength':     1.0,
    'min_data_in_leaf':    1,
}

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2),
    study_name='catboost_f1_pit',
)
study.enqueue_trial(baseline_params)
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SEC, show_progress_bar=True, gc_after_trial=True)

print('best AUC (5-fold):', round(study.best_value, 5))
print('best params:', study.best_params)

## Study summary

In [ ]:
trials_df = study.trials_dataframe(attrs=('number', 'value', 'state', 'params', 'duration'))
trials_df = trials_df.sort_values('value', ascending=False)
trials_df.head(15)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.figsize'] = (12, 4)

vals = [t.value for t in study.trials if t.value is not None]
running_best = np.maximum.accumulate(vals)
plt.plot(vals, '.', alpha=0.5, label='trial AUC')
plt.plot(running_best, label='running best')
plt.xlabel('trial')
plt.ylabel('AUC (5-fold mean)')
plt.legend(); plt.grid(alpha=0.3); plt.title('Optuna search progress'); plt.show()

## Validate best params on the full 10-fold CV

Same splitter / seed as `catboost.ipynb` so the score is directly comparable. After this finishes, paste `study.best_params` into `catboost.ipynb` and re-run it — that produces the OOF/test CSVs the blender consumes.

In [ ]:
best_params = {
    'iterations': 5000,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'task_type': 'GPU',
    'devices': '0',
    'random_seed': 123,
    'verbose': 0,
    **study.best_params,
}

sgkf_full = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=123)
scores, best_iters = [], []

for fold, (tr_idx, val_idx) in enumerate(sgkf_full.split(X, strat_y, groups=groups)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = CatBoostClassifier(**best_params)
    model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        cat_features=categorical_features,
        sample_weight=sample_weights[tr_idx],
        early_stopping_rounds=50,
        verbose=0,
    )
    proba = model.predict_proba(X_val)[:, 1]
    score = roc_auc_score(y_val, proba)
    scores.append(score)
    best_iters.append(model.best_iteration_)
    print(f'Fold {fold}: AUC={score:.5f}  best_iter={model.best_iteration_}')

print(f'\nMean AUC: {np.mean(scores):.5f} ± {np.std(scores):.5f}')
print(f'Mean best_iter: {int(np.mean(best_iters))}  Max best_iter: {int(np.max(best_iters))}')

In [ ]:
with open('./archive/catboost_best_params.json', 'w') as f:
    json.dump({
        'best_params': study.best_params,
        'best_value_5fold': study.best_value,
        'mean_auc_10fold': float(np.mean(scores)),
        'std_auc_10fold':  float(np.std(scores)),
        'best_iters': [int(b) for b in best_iters],
    }, f, indent=2)
print('saved ./archive/catboost_best_params.json')